# CIFAR-10 RGB vs Grayscale Analysis using Lightweight CNN
This notebook explores the performance difference of a Lightweight CNN when trained on standard RGB CIFAR-10 images versus Grayscale images. We use TensorBoard to monitor and compare the metrics.

In [1]:
import torch
from torchvision import transforms
import time
import os

from helper import *

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', device)


PyTorch version: 2.12.0+cpu
Device: cpu


## 3. Data Preparation
CIFAR-10 is naturally RGB. We will create two identical dataloaders, one applying Grayscale conversion, and conditionally avoid re-downloading the dataset.

In [ ]:
IMAGE_SIZE = 32
BATCH_SIZE = 128
NUM_WORKERS = os.cpu_count() or 4


transform_rgb = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


print("Building dataloaders...")
train_loader_rgb, test_loader_rgb, class_names = create_dataloaders(transform_rgb, BATCH_SIZE, NUM_WORKERS)
train_loader_gray, test_loader_gray, _ = create_dataloaders(transform_gray, BATCH_SIZE, NUM_WORKERS)

num_classes = len(class_names)
print('Classes:', class_names)


Building dataloaders...
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 5. Experiment: RGB vs Grayscale

In [3]:
def run_exper(name, model, train_loader, test_loader, epochs=1):
    log_dir = os.path.join('runs', f'cifar10_{name}_{int(time.time())}')
    os.makedirs(log_dir, exist_ok=True)
    set_seed(42)  # re-set seed for deterministic execution within the thread
    _, acc = run_training(model, train_loader, test_loader, log_dir, device, IMAGE_SIZE, epochs)
    return name, acc


## 6. Experiment: Custom Grayscale Only
Run the training experiment specifically on the new custom grayscale images.

In [ ]:

class LinearCombineChannel(object):
    def __init__(self, w_r, w_g, w_b):
        self.w_r = w_r
        self.w_g = w_g
        self.w_b = w_b

    def __call__(self, img):
        r = img[0]
        g = img[1]
        b = img[2]
        combined = self.w_r * r + self.w_g * g + self.w_b * b
        return combined.unsqueeze(0)

# Define transformations for 32x32 training images using the selected combinations
transform_custom_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.36, 0.28, 0.36),
    transforms.Normalize((0.5,), (0.5,))
])

transform_custom_exclusive_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.402, 0.255, 0.342),
    transforms.Normalize((0.5,), (0.5,))
])

print("Building dataloaders for Custom Grayscale experiments...")
#train_loader_custom_gray, test_loader_custom_gray, _ = create_dataloaders(transform_custom_gray, BATCH_SIZE, NUM_WORKERS)
train_loader, test_loader= create_dataloaders(transform_rgb,BATCH_SIZE, NUM_WORKERS)
#train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray, _ = create_dataloaders(transform_custom_exclusive_gray, BATCH_SIZE, NUM_WORKERS)

"""
custom_experiments = [
    ("CustomGray", create_model_single_channel(num_classes), train_loader_custom_gray, test_loader_custom_gray),
    ("CustomExclusiveGray", create_model_single_channel(num_classes), train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray)
]
"""


Building dataloaders for Custom Grayscale experiments...


'\ncustom_experiments = [\n    ("CustomGray", create_model_single_channel(num_classes), train_loader_custom_gray, test_loader_custom_gray),\n    ("CustomExclusiveGray", create_model_single_channel(num_classes), train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray)\n]\n'

In [ ]:
torch.set_num_threads(8)
torch.set_num_interop_threads(8)


name, acc = run_exper(
    "mini-test",
    UltraTinyCNN(),
    train_loader,
    test_loader,
    1
)

print("\nFinal Results:")
print(f" - {name}: {acc:.4f}")

In [7]:
class LinearCombineChannel(object):
    def __init__(self, w_r, w_g, w_b):
        self.w_r = w_r
        self.w_g = w_g
        self.w_b = w_b

    def __call__(self, img):
        r = img[0]
        g = img[1]
        b = img[2]
        combined = self.w_r * r + self.w_g * g + self.w_b * b
        return combined.unsqueeze(0)

# Define transformations for 32x32 training images using the selected combinations
transform_custom_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.36, 0.28, 0.36),
    transforms.Normalize((0.5,), (0.5,))
])

transform_custom_exclusive_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.402, 0.255, 0.342),
    transforms.Normalize((0.5,), (0.5,))
])

print("Building dataloaders for Custom Grayscale experiments...")
train_loader_custom_gray, test_loader_custom_gray, _ = create_dataloaders(transform_custom_gray, BATCH_SIZE, NUM_WORKERS)
#train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray, _ = create_dataloaders(transform_custom_exclusive_gray, BATCH_SIZE, NUM_WORKERS)

custom_experiments = [
    ("CustomGray", create_model_single_channel(num_classes), train_loader_custom_gray, test_loader_custom_gray),
    ("CustomExclusiveGray", create_model_single_channel(num_classes), train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray)
]

custom_results = {}
print("\nStarting Experimets concurrently: Custom Grayscale and Custom Exclusive Grayscale...")

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(run_exper, *exp, epochs=10): exp[0] for exp in custom_experiments}
    for future in concurrent.futures.as_completed(futures):
        name = futures[future]
        try:
            exp_name, acc = future.result()
            custom_results[exp_name] = acc
            print(f"Finished {exp_name} - Test Acc: {acc:.4f}")
        except Exception as e:
            print(f"Experiment {name} generated an exception: {e}")

print("\nFinal Results:")
for name, acc in custom_results.items():
    print(f" - {name}: {acc:.4f}")

Building dataloaders for Custom Grayscale experiments...


NameError: name 'train_loader_custom_exclusive_gray' is not defined

In [ ]:
class LinearCombineChannel(object):
    def __init__(self, w_r, w_g, w_b):
        self.w_r = w_r
        self.w_g = w_g
        self.w_b = w_b

    def __call__(self, img):
        r = img[0]
        g = img[1]
        b = img[2]
        combined = self.w_r * r + self.w_g * g + self.w_b * b
        return combined.unsqueeze(0)

# Define transformations for 32x32 training images using the selected combinations
transform_custom_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.36, 0.28, 0.36),
    transforms.Normalize((0.5,), (0.5,))
])



In [ ]:
#tensorboard --logdir runs --bind_all 